# Camada Bronze - DataCo Smart Supply Chain + US Holiday Dates (Diogo)

Carrega os ficheiros originais para o HDFS e cria as tabelas Bronze em formato Delta, na stack comum do grupo.
Os dados ficam tal como vêm da fonte (todas as colunas como texto, com os nomes originais),
com os metadados de ingestão `_data_ingestao` e `_ficheiro_origem`.

| Dataset | Ficheiro | Tabela Bronze |
|---|---|---|
| DataCo Smart Supply Chain (principal) | DataCoSupplyChainDataset.csv | bronze.dataco_supply_chain |
| US Holiday Dates (complementar) | US Holiday Dates (2004-2021).csv | bronze.us_holidays |

Fontes: Kaggle `shashwatwork/dataco-smart-supply-chain-for-big-data-analysis` (ficheiro em ISO-8859-1 / Latin-1) e
`donnetew/us-holiday-dates-2004-2021`. Os feriados são também usados pelo Henrique (KPI 3) e pelo Quévin (época de festas).

In [1]:
ficheiros = {
    "/home/jovyan/work/data/diogo/DataCoSupplyChainDataset.csv": "/projeto/bronze/diogo/DataCoSupplyChainDataset.csv",
    "/home/jovyan/work/data/holidays/US Holiday Dates (2004-2021).csv": "/projeto/bronze/diogo/us_holiday_dates_2004_2021.csv",
}

In [2]:
# Copiar os ficheiros originais para a zona Bronze do HDFS (full load: apaga a versão anterior e volta a carregar)
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

for from_path, to_path in ficheiros.items():
    client.delete(to_path)
    client.upload(to_path, from_path)
    print(to_path, client.status(to_path)["length"], "bytes")

/projeto/bronze/diogo/DataCoSupplyChainDataset.csv 95910149 bytes
/projeto/bronze/diogo/us_holiday_dates_2004_2021.csv 15699 bytes


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Bronze - DataCo + US Holiday Dates (Diogo)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Os nomes originais das colunas têm espaços e caracteres especiais;
# o column mapping do Delta permite guardá-los tal como vêm da fonte
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db/'")

def gravar_bronze(df, tabela):
    """Acrescenta os metadados de ingestão e grava a tabela Delta tal como veio da fonte."""
    df = df.withColumn("_data_ingestao", F.current_timestamp()) \
           .withColumn("_ficheiro_origem", F.input_file_name())
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/bronze.db/{tabela}/") \
        .saveAsTable(f"bronze.{tabela}")
    print(f"bronze.{tabela}: {spark.table(f'bronze.{tabela}').count()} linhas, {len(df.columns) - 2} colunas originais")

def ler_csv(hdfs_path, **opcoes):
    """Lê um CSV do HDFS com todas as colunas como texto (sem inferSchema)."""
    leitor = spark.read.option("header", "true").option("multiLine", "true").option("escape", '"')
    for k, v in opcoes.items():
        leitor = leitor.option(k, v)
    return leitor.csv("hdfs://hdfs-nn:9000" + hdfs_path)

In [5]:
gravar_bronze(ler_csv("/projeto/bronze/diogo/DataCoSupplyChainDataset.csv", encoding="ISO-8859-1"), "dataco_supply_chain")
gravar_bronze(ler_csv("/projeto/bronze/diogo/us_holiday_dates_2004_2021.csv"), "us_holidays")

bronze.dataco_supply_chain: 180519 linhas, 53 colunas originais
bronze.us_holidays: 342 linhas, 6 colunas originais


## Verificação

As tabelas Bronze têm de ter o mesmo número de linhas dos ficheiros originais.

In [6]:
import csv
def linhas(caminho, encoding):
    with open(caminho, encoding=encoding, newline="") as f:
        return sum(1 for _ in csv.reader(f)) - 1

origem = {"dataco_supply_chain": ("/home/jovyan/work/data/diogo/DataCoSupplyChainDataset.csv", "ISO-8859-1"),
          "us_holidays": ("/home/jovyan/work/data/holidays/US Holiday Dates (2004-2021).csv", "utf-8")}
for tabela, (caminho, enc) in origem.items():
    n, real = linhas(caminho, enc), spark.table(f"bronze.{tabela}").count()
    print(f"{tabela:20s} ficheiro {n:>7}  bronze {real:>7}  {'OK' if n == real else 'DIFERENTE'}")
    assert n == real
print("colunas originais do DataCo:", len(spark.table("bronze.dataco_supply_chain").columns) - 2)

dataco_supply_chain  ficheiro  180519  bronze  180519  OK
us_holidays          ficheiro     342  bronze     342  OK
colunas originais do DataCo: 53


In [7]:
spark.table("bronze.dataco_supply_chain").select("Order Id", "Order Item Id", "order date (DateOrders)", "Order Country", "Sales").show(5, truncate=False)
spark.table("bronze.us_holidays").show(5, truncate=False)

+--------+-------------+-----------------------+-------------+------+
|Order Id|Order Item Id|order date (DateOrders)|Order Country|Sales |
+--------+-------------+-----------------------+-------------+------+
|77202   |180517       |1/31/2018 22:56        |Indonesia    |327.75|
|75939   |179254       |1/13/2018 12:27        |India        |327.75|
|75938   |179253       |1/13/2018 12:06        |India        |327.75|
|75937   |179252       |1/13/2018 11:45        |Australia    |327.75|
|75936   |179251       |1/13/2018 11:24        |Australia    |327.75|
+--------+-------------+-----------------------+-------------+------+
only showing top 5 rows

+----------+-----------+---------+-----+---+----+--------------------------+-----------------------------------------------------------------------+
|Date      |Holiday    |WeekDay  |Month|Day|Year|_data_ingestao            |_ficheiro_origem                                                       |
+----------+-----------+---------+-----+---+---

In [8]:
spark.stop()